# Fresh Kaggle T4 x2 Run: Fix 5 + Fix 11 Only

Use a fresh Kaggle notebook/session with **Internet ON** and **Accelerator: T4 x2**. This notebook deliberately does not run Fix 1, Fix 2, Fix 3, or Fix 4.

Expected output file: `/kaggle/working/fix5_11_t4x2_outputs.zip`.

Pip may print `google-colab` or `opentelemetry` dependency warnings. Those are okay if the cell continues and prints `Successfully installed` or reaches the next section.

In [ ]:
import os
import pathlib
import subprocess
import time

WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'rag-hallucination-detection'
REMOTE = 'https://github.com/Saket-Maganti/rag-hallucination-detection.git'

def run(cmd, cwd=None, check=True):
    print('\n>>>', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check)

print('START', time.ctime(), flush=True)
run(['bash', '-lc', 'pkill -f kaggle_fix5_11_t4x2 || true; pkill -f kaggle_stream_fix5_11_t4x2 || true; pkill -x ollama || true'], check=False)
run(['nvidia-smi'], check=False)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--progress', '--branch', 'main', REMOTE, str(REPO)], cwd=WORK)
else:
    run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=False)
    run(['git', '-C', str(REPO), 'checkout', 'main'], check=False)
    run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=False)

os.chdir(REPO)
run(['git', 'log', '-1', '--oneline'])
run(['bash', '-lc', 'test -f scripts/kaggle_fix5_11_t4x2.sh && test -f scripts/kaggle_stream_fix5_11_t4x2.py && echo scripts OK'])
print('Repo ready at', REPO, flush=True)

## 1. Setup + Ollama Preflight

This installs dependencies, installs/verifies Ollama, starts two Ollama servers, pulls `mistral`, and checks both ports. Wait for `preflight OK` before running the next cell.

In [ ]:
!python -u scripts/kaggle_stream_fix5_11_t4x2.py --stage setup --heartbeat 15

## 2. Run Fix 5 + Fix 11 in Parallel

Fix 5 runs on GPU0 / Ollama port 11434. Fix 11 runs on GPU1 / Ollama port 11435. Expected wall time after setup: roughly 2.5-4.5 hours.

In [ ]:
!python -u scripts/kaggle_stream_fix5_11_t4x2.py --stage parallel --heartbeat 30

## 3. Status

Use this after the parallel run, or after any interruption. It only prints row counts and file paths.

In [ ]:
!python -u scripts/kaggle_stream_fix5_11_t4x2.py --stage status --heartbeat 15

## 4. Package Outputs

Download `/kaggle/working/fix5_11_t4x2_outputs.zip` from the Kaggle right sidebar after this completes.

In [ ]:
!python -u scripts/kaggle_stream_fix5_11_t4x2.py --stage package --heartbeat 15
!ls -lh /kaggle/working/fix5_11_t4x2_outputs.zip

## Debug If Setup Fails

Run this only if setup fails.

In [ ]:
!echo '--- git ---'
!git -C /kaggle/working/rag-hallucination-detection log -1 --oneline || true
!echo '--- ollama binaries ---'
!which ollama || true
!ls -lh /usr/local/bin/ollama /usr/bin/ollama 2>/dev/null || true
!echo '--- processes ---'
!ps aux | grep -E 'ollama|fix5_11|kaggle_stream' | grep -v grep || true
!echo '--- gpu ---'
!nvidia-smi || true
!echo '--- ollama gpu0 log ---'
!tail -n 160 /kaggle/working/fix5_11_t4x2_logs/ollama_gpu0.log 2>/dev/null || true
!echo '--- ollama gpu1 log ---'
!tail -n 160 /kaggle/working/fix5_11_t4x2_logs/ollama_gpu1.log 2>/dev/null || true